In [ ]:
#| hide
from programming_first_principles.core import prepare_notebook
from collections.abc import Sequence, MutableSequence
import sys

In [ ]:
#| hide
prepare_notebook()

# Bytes, Bytearray, and Memoryview
* A byte sequence is a contiguous sequence of 8-bit unsigned integers (`0` to `255`). Unlike `str`, raw bytes carry no inherent text meaning on their own.
* Python provides `bytes` (immutable sequence of bytes), `bytearray` (mutable sequence of bytes), and `memoryview` (zero-copy view over a byte buffer).

## `bytes` — Immutable Sequence of Bytes
* `b'...'` literals or the `bytes(...)` constructor create an immutable `bytes` object.
* Indexing a single position `b[i]` returns an `int` in `0..255`, whereas slicing `b[i:j]` returns a `bytes` subsequence.

#### `bytes` Literals and `bytes()` Constructor

In [ ]:
bytes_literal = b'Text'
f'{bytes_literal = }, {type(bytes_literal) = }'
f'{isinstance(bytes_literal, Sequence) = }'
f'{sys.getsizeof(bytes_literal) = }'

"bytes_literal = b'Text', type(bytes_literal) = <class 'bytes'>"

'isinstance(bytes_literal, Sequence) = True'

'sys.getsizeof(bytes_literal) = 37'

In [ ]:
bytes_from_ints = bytes([84, 101, 120, 116])
f'{bytes_from_ints = }, {bytes_literal == bytes_from_ints = }'

"bytes_from_ints = b'Text', bytes_literal == bytes_from_ints = True"

#### Indexing Returns `int` (`0..255`), Slicing Returns `bytes`

In [ ]:
f'{bytes_literal[0] = }, {type(bytes_literal[0]) = }'
f'{bytes_literal[0:2] = }, {type(bytes_literal[0:2]) = }'
f'{list(bytes_literal) = }'

"bytes_literal[0] = 84, type(bytes_literal[0]) = <class 'int'>"

"bytes_literal[0:2] = b'Te', type(bytes_literal[0:2]) = <class 'bytes'>"

'list(bytes_literal) = [84, 101, 120, 116]'

#### ASCII-Only Restriction on `b'...'` Literals
* A `b'...'` literal only accepts ASCII characters (`0x00`..`0x7F`) or hex escapes (`\xhh`).
* Non-ASCII characters in a `b'...'` literal raise a `SyntaxError`, and `\U........` escapes inside `b'...'` are kept as literal backslash characters rather than Unicode code points.

In [ ]:
try:
    eval("b'Python 3 - 🐍'")
except SyntaxError as e:
    f'{e = }'

'e = SyntaxError(\'bytes can only contain ASCII literal characters\', (\'<string>\', 1, 1, "b\'Python 3 - 🐍\'", 1, 16))'

In [ ]:
raw_escape_bytes = rb'\U0001F40D'
f'{raw_escape_bytes = }, {len(raw_escape_bytes) = }'
f'{b"\xf0\x9f\x90\x8d" = }, {len(b"\xf0\x9f\x90\x8d") = }'

"raw_escape_bytes = b'\\\\U0001F40D', len(raw_escape_bytes) = 10"

'b"\\xf0\\x9f\\x90\\x8d" = b\'\\xf0\\x9f\\x90\\x8d\', len(b"\\xf0\\x9f\\x90\\x8d") = 4'

#### `str` vs `bytes`: Distinct Types
* A `str` of characters and a `bytes` of raw 8-bit values are never equal (`'Text' == b'Text'` is `False`), and they cannot be concatenated directly without an explicit conversion.

In [ ]:
normal_string = 'Text'
bytes_string = b'Text'

f'{normal_string == bytes_string = }'
try:
    b'Python 3 ' + ' - 🐍'
except TypeError as e:
    f'{e = }'

'normal_string == bytes_string = False'

'e = TypeError("can\'t concat str to bytes")'

#### Immutability of `bytes`

In [ ]:
b1 = b'\xf0\x9f\x98\x80'
b2 = b1
b3 = b1 + b'ABC'

f'{id(b1) = }, {id(b2) = }, {b1 is b2 = }'
f'{id(b3) = }, {b3 = }'

'id(b1) = 4430514672, id(b2) = 4430514672, b1 is b2 = True'

"id(b3) = 4430850272, b3 = b'\\xf0\\x9f\\x98\\x80ABC'"

In [ ]:
try:
    b1[0] = 65
except TypeError as e:
    f'{e = }'

'e = TypeError("\'bytes\' object does not support item assignment")'

## `bytearray` — Mutable Sequence of Bytes
* `bytearray` is the mutable counterpart to `bytes` and implements `MutableSequence` (`__setitem__`, `__delitem__`, `append`, `extend`).

In [ ]:
data = bytearray(b'Python Conference')
f'{data = }, {isinstance(data, MutableSequence) = }'

"data = bytearray(b'Python Conference'), isinstance(data, MutableSequence) = True"

In [ ]:
before_id = id(data)
data[0] = ord('C')
data.append(ord('!'))
f'{data = }, {id(data) == before_id = }'

"data = bytearray(b'Cython Conference!'), id(data) == before_id = True"

## `memoryview` — Zero-Copy Buffer View
* Slicing a `bytes` or `bytearray` allocates and copies a new sequence.
* `memoryview(obj)` exposes the buffer protocol directly: slicing a `memoryview` creates a new view into the **same underlying memory** without copying bytes, and writing through a slice of a writable `memoryview` mutates the original buffer in place.

In [ ]:
data = bytearray(b'Python Conference')
view = memoryview(data)
my_slice = view[0:6]

f'{view = }'
f'{my_slice = }, {my_slice.tobytes() = }'

'view = <memory>'

"my_slice = <memory>, my_slice.tobytes() = b'Python'"

In [ ]:
view[0:4] = b'DOGE'
f'{data = }'
f'{my_slice.tobytes() = }'

"data = bytearray(b'DOGEon Conference')"

"my_slice.tobytes() = b'DOGEon'"

## References
* [PEP 3137 – Immutable Bytes and Mutable Buffer (Guido van Rossum)](https://peps.python.org/pep-3137/)
* [PEP 3118 – Revising the Buffer Protocol (Travis Oliphant, Carl Banks)](https://peps.python.org/pep-3118/)
* [Python Library Reference – Binary Sequence Types: `bytes`, `bytearray`, `memoryview`](https://docs.python.org/3/library/stdtypes.html#binary-sequence-types-bytes-bytearray-memoryview)